# PT01: Introduction to PyTorch: solutions

The solutions to the exercises of [PT01](https://colab.research.google.com/github/sscardapane/teaching-materials/blob/main/notebooks/PT01_Introduction_to_PyTorch.ipynb). Each solution comes under the heading it has in the lab, with a short explanation followed by the code. If you fall behind during the lab, copy the solution cell into your notebook in place of your version and continue from there.

## Before PT02: a short check

1. Reducing over `dim=1` gives shape `(3,)`, which does not broadcast against `(3, 2)`. Use `A.mean(dim=1, keepdim=True)`, with shape `(3, 1)`. With the square `S`, the row means `[2, 6]` have shape `(2,)` and broadcast as a **row**: the first mean is subtracted from the first column and the second from the second column. The result `[[-1, -3], [3, 1]]` has the right shape and no error, but the correct answer is `[[-1, 1], [-1, 1]]`. Square shapes hide axis mistakes.
2. `[2, 4] + [3, 3] = [5, 7]`, the gradient of `a.square().sum() + (3 * a).sum()`: `backward()` adds, so the stored gradient belongs to the sum of the two functions. After the update `a = [0.5, 1.3]`, and the third call adds `[1.0, 2.6]`, giving `[6.0, 9.6]` instead of `[1.0, 2.6]`. Set `a.grad = None` before each backward call whose gradient we want on its own.
3. Logits `(5, 3)`; mean loss `()`; weight gradient `(3, 4)`; bias gradient `(3,)`. With `reduction="none"` the loss has shape `(5,)` and `backward()` raises an error, because it needs a scalar to start from. Calling `.mean()` first gives the same gradients as before.

The code below checks the three answers.

In [ ]:
# Executable answers to the check.
S = torch.tensor([[1., 3.], [5., 7.]])
torch.testing.assert_close(S - S.mean(dim=1), torch.tensor([[-1., -3.], [3., 1.]]))
torch.testing.assert_close(S - S.mean(dim=1, keepdim=True), torch.tensor([[-1., 1.], [-1., 1.]]))

a = torch.tensor([1., 2.], requires_grad=True)
a.square().sum().backward()
(3 * a).sum().backward()
torch.testing.assert_close(a.grad, torch.tensor([5., 7.]))
with torch.no_grad():
    a -= 0.1 * a.grad
a.square().sum().backward()
torch.testing.assert_close(a, torch.tensor([0.5, 1.3]))
torch.testing.assert_close(a.grad, torch.tensor([6.0, 9.6]))

check_X, check_y = torch.randn(5, 4), torch.tensor([0, 2, 1, 1, 0])
check_W = torch.randn(3, 4, requires_grad=True)
check_b = torch.zeros(3, requires_grad=True)
per_example = F.cross_entropy(check_X @ check_W.T + check_b, check_y, reduction="none")
assert per_example.shape == (5,)
try:
    per_example.backward()
except RuntimeError as error:
    print("Non-scalar backward:", str(error).splitlines()[0])
else:
    raise AssertionError("backward() on a vector loss should fail")
per_example.mean().backward()
assert check_W.grad.shape == (3, 4) and check_b.grad.shape == (3,)
mean_W_grad, mean_b_grad = torch.autograd.grad(
    F.cross_entropy(check_X @ check_W.T + check_b, check_y), (check_W, check_b))
torch.testing.assert_close(check_W.grad, mean_W_grad)
torch.testing.assert_close(check_b.grad, mean_b_grad)

## Optional: tensor semantics challenges

### 1. One error per example ★

Broadcasting `(3, 1)` against `(3,)` produces a `(3, 3)` matrix: entry `(i, j)` compares prediction `i` with target `j`. Its rows are `[1, -1, -4]`, `[2, 0, -3]` and `[4, 2, -1]`, and the mean of the nine squares is `52/9`. The intended errors are `[1, 0, -1]`, with MSE `2/3`. Both losses are scalars, so checking the shape of the loss does not catch the bug; checking the shape of the errors does. `squeeze(-1)` removes only the last axis, so a batch with a single example keeps its batch axis.

In [ ]:
errors = predictions.squeeze(-1) - targets
mse = errors.square().mean()
assert errors.shape == targets.shape == (3,)
torch.testing.assert_close(errors, torch.tensor([1., 0., -1.]))
torch.testing.assert_close(mse, torch.tensor(2. / 3.))

### 2. Which axis contains the examples? ★★

`batch.T @ weights` combines the first feature of every example with the first weight, and so on, which gives `[31, 42]`. The intended predictions are `[21, 43]`. The batch is square, so both results have shape `(2,)`: only a check on the values rejects the original. With three examples, `larger_batch.T` has shape `(2, 3)` and its product with `weights`, which has size two, raises a dimension mismatch.

In [ ]:
row_predictions = batch @ weights
assert row_predictions.shape == (batch.shape[0],)
torch.testing.assert_close(row_predictions, torch.tensor([21., 43.]))

larger_batch = torch.cat([batch, torch.tensor([[5., 6.]])], dim=0)
larger_predictions = larger_batch @ weights
assert larger_predictions.shape == (3,)
torch.testing.assert_close(larger_predictions, torch.tensor([21., 43., 65.]))

### 3. One distribution per example ★★

`softmax(dim=0)` normalizes each column, so the three examples compete for each class. The columns sum to one, while the rows sum to about `1.05`, `0.80` and `1.15`, and every entry is still between 0 and 1. The first example is predicted as class 1 (probability `0.58`) although its largest logit is class 0: normalizing over the classes gives `[0.84, 0.11, 0.04]`. `dim=-1` names the class axis even if we later add more leading axes. For predictions alone the softmax is not needed: `scores.argmax(dim=-1)` gives the same classes.

In [ ]:
probabilities = scores.softmax(dim=-1)
assert probabilities.shape == scores.shape
torch.testing.assert_close(probabilities.sum(dim=-1), torch.ones(3))
torch.testing.assert_close(probabilities.argmax(dim=-1), scores.argmax(dim=-1))
torch.testing.assert_close(probabilities.argmax(dim=-1), torch.tensor([0, 2, 0]))

## Optional: autograd detective

### 1. Did the gradient stop at the intermediate tensor? ★★

The gradient did pass through `hidden`. `leaf` is a leaf, with `grad_fn=None` and gradient `[18, 36]`. `hidden` is not a leaf and has a backward node: its gradient `[6, 12]` was computed along the way, but not stored, which is what the warning says. To keep it, call `retain_grad()` before `backward()`, on a fresh computation.

In [ ]:
leaf = torch.tensor([1., 2.], requires_grad=True)
hidden = 3 * leaf
hidden.retain_grad()
hidden.square().sum().backward()
assert leaf.is_leaf and not hidden.is_leaf
torch.testing.assert_close(leaf.grad, torch.tensor([18., 36.]))
torch.testing.assert_close(hidden.grad, torch.tensor([6., 12.]))

### 2. Can we reconnect a detached tensor? ★

`backward()` succeeds, but on the wrong leaf: `restart.grad` is `[6, 12]`, while `original.grad` stays `None`. Turning gradients back on after `detach()` creates a new leaf; it does not restore the operation that connected the two tensors. The fix keeps that operation in the recorded computation.

In [ ]:
original = torch.tensor([1., 2.], requires_grad=True)
connected = 3 * original
connected.square().sum().backward()
torch.testing.assert_close(original.grad, torch.tensor([18., 36.]))

### 3. Is accumulation always a mistake? ★★

Here accumulation is intended. The gradient is `10/3` after the first call and `28/3` after the second, the same as one `backward()` on the full batch. Each microbatch contributes its sum of squared errors divided by the total number of examples, so clearing between the calls would throw away the first contribution. Dividing each microbatch mean by two instead gives `11.5`: the microbatch with one example then counts as much as the one with two.

In [ ]:
full_parameter = torch.tensor(1., requires_grad=True)
all_values = torch.cat(microbatches)
(full_parameter * all_values).square().mean().backward()
torch.testing.assert_close(parameter.grad, full_parameter.grad)
torch.testing.assert_close(parameter.grad, torch.tensor(28. / 3.))

### 4. Is every in-place change safe inside `no_grad`? ★★★

`backward()` raises an error. The in-place update changed a value that backward needs (the square needs the original `mutable`); `no_grad` only means that the update itself is not recorded. Move the update after `backward()`. Modifying `mutable.detach()` in place changes the same memory and fails in the same way.

In [ ]:
mutable = torch.tensor([1., 2.], requires_grad=True)
mutable.square().sum().backward()
torch.testing.assert_close(mutable.grad, torch.tensor([2., 4.]))
with torch.no_grad():
    mutable -= 0.1 * mutable.grad
mutable.grad = None
torch.testing.assert_close(mutable, torch.tensor([0.8, 1.6]))
assert mutable.is_leaf and mutable.requires_grad

## Optional: a few array operations

### The two diagonals ★

`A.diagonal()` returns the main diagonal. Flipping the columns with `A.flip(dims=[1])` turns the anti-diagonal into the main diagonal, so the same call returns it. For an odd size the center element appears in both sums, as required.

In [ ]:
def sum_diagonals(A: Float[torch.Tensor, "n n"]) -> Optional[Float[torch.Tensor, ""]]:
    return A.diagonal().sum() + A.flip(dims=[1]).diagonal().sum()

### Normalizing the rows ★

We reduce over `dim=1`, the features of each row, with `keepdim=True`, so the means and standard deviations have shape `(batch, 1)` and broadcast against `A`. The standard deviation of a constant row is zero. `clamp_min(eps)` replaces it with `eps`, so the centered row, which is all zeros, stays zero instead of becoming `0/0 = nan`.

In [ ]:
def normalize_rows(A: Float[torch.Tensor, "batch features"], eps: float = 1e-8) -> Optional[Float[torch.Tensor, "batch features"]]:
    centered = A - A.mean(dim=1, keepdim=True)
    std = A.std(dim=1, correction=0, keepdim=True)
    return centered / std.clamp_min(eps)

### Cosine similarity ★★

`F.normalize` divides each row by its Euclidean norm, so the dot product of two normalized rows is their cosine. `unit @ unit.T` computes all the pairs at once, with shape `(batch, batch)`. `F.normalize` divides by the larger of the norm and `eps`, so a zero row stays zero and all its similarities are zero.

In [ ]:
def cosine_similarity(A: Float[torch.Tensor, "batch features"]) -> Optional[Float[torch.Tensor, "batch batch"]]:
    unit = F.normalize(A, p=2, dim=1, eps=1e-12)
    return unit @ unit.T